# TVAM AID: Metric Trimming Demonstration

## 1. Problem Definition

The commonly used metrics of Process Window (PW), In-Part Dose-Range (IPDR), and Voxel Error Rate (VER) depend on the extreme values of the in-part and out-of-part dose distributions. Thus, it is possible a small number of voxels with unusually high or low achieved doses can have a disproportionate influence on the reported metrics.

This notebook investigates the sensitivity of the metrics to those `outlier' doses. Increasing percentages of voxels are excluded equally from the upper and lower tails of the dose distributions thus replacing the extrema used by the metrics with corresponding percentile values. This does not change the achieved dose field as trimming is applied only during post-processing of the metrics. 

The percentile-trimmed metrics are used here as an initial exploration of metric robustness and are not proposed as replacements for the untrimmed definitions.

Note: VER is not considered here as. Whilst VER can be affected by outlier dose values, the threshold parameter sweeps for the geometries used in the publication show that a perfect VER of zero is readily achievable by many threshold pairs. Thus, percentile trimming would have no effect on the VER value.

## 2. Configuration

In [1]:
from tvam_aid import RunConfig, Thresholds, loadTargetGeometry, showTarget, \
    nonNegativityConstraint, runOptimisation, printMetrics, \
    calcTrimmedMetrics, plotMetricSensitivity, \
    showAchievedDose3D, plotHistogram, plotSliceMetrics
from tvam_aid.penalties import L2N, OSP, OSPW
from tvam_aid.operators import createLinearOperator
from tvam_aid.callbacks import RecordMetrics, Progress
import numpy as np

In [2]:
# select the target geometry
pathToTarget = '../targets/gyroid.npy'

# select which approach to use
approach = OSPW(w=0.0)

# define settings for this run
config = RunConfig(
    device = 'gpu',
    numIters = 1500,
    angles = np.linspace(0, 360, 360*4, endpoint=False)
)

# define the upper and lower thresholds
thresholds = Thresholds(
    lower = 0.7,
    upper = 0.9
)

# define which callbacks to use in the optimisation
metrics = RecordMetrics()
progress = Progress()

## 3. Load Target Geometry and Construct the TVAM Operator

In [3]:
# load target geometry
target = loadTargetGeometry(pathToTarget)
showTarget(target)

# construct the TVAM operator
tvamOperator = createLinearOperator(target, config)

## 4. Run Optimisation

In [ ]:
# define the optimisation problem
objective = approach.build(tvamOperator, target, thresholds)
constraint = nonNegativityConstraint()

# solve the optimisation
metricsCallback = metrics.build(target, thresholds, tvamOperator)
progressCallback = progress.build()
callbacks = [metricsCallback, progressCallback]
illuminationPlan, achievedDose = runOptimisation(objective, constraint, target, tvamOperator, config, callbacks)

  0%|          | 0/1500 [00:00<?, ?it/s]

In [ ]:
printMetrics(metricsCallback)
showAchievedDose3D(achievedDose)
fig1, ax1 = plotHistogram(achievedDose, target, thresholds)

## 5. Calculate Metrics with Increased Trimming

In [ ]:
# define the total percentages of each distribution to remove
alphas = np.linspace(0.0, 4, num=101)

# calculate the trimmed metrics
trimmedIPDR, trimmedPW = calcTrimmedMetrics(achievedDose.array, target, alphas)

## 6. Visualise Metric Sensitivity to Trimming

In [ ]:
fig, ax = plotMetricSensitivity(alphas, trimmedIPDR, trimmedPW, highlightedPercentage=0.05)

## 7. Visualise Slice-Wise Sensitivity to Trimming

In [ ]:
alphas = [0.025, 0.05]
fig, ax = plotSliceMetrics(achievedDose, target, alphas)